In [1]:
import os, random, math, time
import pandas as pd
import numpy as np
from collections import Counter
try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import nltk
from pyvi import ViTokenizer
import matplotlib.pyplot as plt
import sacrebleu

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [2]:
BASE_DIR = r"c:\Users\Acer\Downloads\Project NLP"

class Config:
    TRAIN_FILE = r"c:\Users\Acer\Downloads\Project NLP\train_200k.csv"
    VAL_FILE   = r"c:\Users\Acer\Downloads\Project NLP\val.csv"
    TEST_FILE  = r"c:\Users\Acer\Downloads\Project NLP\test.csv"

    # vocabulary
    MIN_FREQ   = 2
    MAX_LEN    = 50        

    # model
    EMB_DIM    = 256
    HID_DIM    = 512
    ENC_LAYERS = 2
    DEC_LAYERS = 2
    DROPOUT    = 0.3

    # training
    BATCH_SIZE      = 32
    LR              = 1e-3
    TEACHER_FORCING = 0.5
    EPOCHS          = 15
    CLIP_GRAD       = 1.0

    # inference
    BEAM_SIZE  = 5
    MAX_DECODE = 50

    # special tokens
    PAD = "<pad>"
    UNK = "<unk>"
    SOS = "<sos>"
    EOS = "<eos>"

    SEED   = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if cfg.DEVICE == "cuda":
    torch.cuda.manual_seed_all(cfg.SEED)

print(f"Device: {cfg.DEVICE}")


Device: cuda


In [3]:
train_df = pd.read_csv(cfg.TRAIN_FILE)
val_df = pd.read_csv(cfg.VAL_FILE)
test_df = pd.read_csv(cfg.TEST_FILE)

def clean_and_filter_data(df, is_train=False, max_samples=800000):
    initial_len = len(df)
    
    # Bước 1: Xóa các dòng bị thiếu dữ liệu (NaN)
    df = df.dropna(subset=['en', 'vi'])
    
    # Bước 2: Xóa các câu trùng lặp y hệt nhau
    df = df.drop_duplicates(subset=['en', 'vi'])
    
    # Bước 3: Đếm số lượng TỪ (Word count) thay vì đếm KÝ TỰ (Length)
    # Tách chuỗi bằng khoảng trắng để đếm số từ
    df['en_word_count'] = df['en'].apply(lambda x: len(str(x).split()))
    df['vi_word_count'] = df['vi'].apply(lambda x: len(str(x).split()))
    
    # Bước 4: Lọc độ dài câu (Chỉ giữ câu từ 2 đến 50 từ)
    # Loại bỏ các câu 1 từ (thường là tên riêng, nhiễu) hoặc quá dài (gây tràn RAM)
    df = df[(df['en_word_count'] >= 2) & (df['en_word_count'] <= 50)]
    df = df[(df['vi_word_count'] >= 2) & (df['vi_word_count'] <= 50)]
    
    # Bước 5: Bộ lọc Tỷ lệ độ dài (Length Ratio Filter)
    # Ngữ pháp Anh-Việt thường có độ dài tương đương. Nếu 1 câu dài gấp 3 lần câu kia -> Dữ liệu lỗi (Misaligned)
    df['length_ratio'] = df['en_word_count'] / df['vi_word_count']
    df = df[(df['length_ratio'] >= 0.5) & (df['length_ratio'] <= 2.0)]
    
    # Bước 6: Rút gọn ngẫu nhiên (Downsampling) - CHỈ ÁP DỤNG CHO TẬP TRAIN
    if is_train and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42)
        
    # Xóa các cột đếm từ vì không cần thiết cho Transformer nữa
    df = df.drop(columns=['en_word_count', 'vi_word_count', 'length_ratio'])
    
    final_len = len(df)
    print(f"Đã giữ lại {final_len:,} / {initial_len:,} câu ({final_len/initial_len*100:.1f}%)")
    
    return df

# Tiến hành dọn dẹp
print("\n Xử lý tập Train:")
train_df = clean_and_filter_data(train_df, is_train=True, max_samples=750000)

print("\n Xử lý tập Validation:")
val_df = clean_and_filter_data(val_df, is_train=False)

print("\n Xử lý tập Test:")

print(f"  test  -> {cfg.TEST_FILE}")
print(f"  val   -> {cfg.VAL_FILE}")
print(f"  train -> {cfg.TRAIN_FILE}")
print("Saved cleaned dataset files:")

clean_test_path = os.path.join(BASE_DIR, "test_clean.csv")
clean_val_path = os.path.join(BASE_DIR, "val_clean.csv")
clean_train_path = os.path.join(BASE_DIR, "train_200k_clean.csv")

test_df.to_csv(clean_test_path, index=False)
val_df.to_csv(clean_val_path, index=False)
train_df.to_csv(clean_train_path, index=False)

cfg.TEST_FILE = clean_test_path
cfg.VAL_FILE = clean_val_path
cfg.TRAIN_FILE = clean_train_path

print(f"  test_clean  -> {cfg.TEST_FILE}")
print(f"  val_clean   -> {cfg.VAL_FILE}")
print(f"  train_clean -> {cfg.TRAIN_FILE}")

# Lưu tập dữ liệu đã được làm sạch và cập nhật cfg để các bước sau dùng file mới
test_df = clean_and_filter_data(test_df, is_train=False)


 Xử lý tập Train:
Đã giữ lại 193,741 / 193,741 câu (100.0%)

 Xử lý tập Validation:
Đã giữ lại 10,974 / 10,974 câu (100.0%)

 Xử lý tập Test:
  test  -> c:\Users\Acer\Downloads\Project NLP\test.csv
  val   -> c:\Users\Acer\Downloads\Project NLP\val.csv
  train -> c:\Users\Acer\Downloads\Project NLP\train_200k.csv
Saved cleaned dataset files:
  test_clean  -> c:\Users\Acer\Downloads\Project NLP\test_clean.csv
  val_clean   -> c:\Users\Acer\Downloads\Project NLP\val_clean.csv
  train_clean -> c:\Users\Acer\Downloads\Project NLP\train_200k_clean.csv
Đã giữ lại 10,868 / 11,225 câu (96.8%)


In [4]:
def tokenize_en(text: str):
    """NLTK word tokeniser for English."""
    return nltk.word_tokenize(text.lower())

def tokenize_vi(text: str):
    """pyvi word-segment tokeniser for Vietnamese."""
    # ViTokenizer.tokenize returns tokens joined by spaces
    return ViTokenizer.tokenize(text).split()

class Vocabulary:
    def __init__(self, min_freq: int = 2):
        self.min_freq = min_freq
        self.stoi = {}   # string → index
        self.itos = {}   # index → string
        for i, tok in enumerate([cfg.PAD, cfg.UNK, cfg.SOS, cfg.EOS]):
            self.stoi[tok] = i
            self.itos[i]   = tok

    @property
    def pad_idx(self): return self.stoi[cfg.PAD]
    @property
    def unk_idx(self): return self.stoi[cfg.UNK]
    @property
    def sos_idx(self): return self.stoi[cfg.SOS]
    @property
    def eos_idx(self): return self.stoi[cfg.EOS]
    def __len__(self):  return len(self.stoi)

    def build(self, token_lists):
        counter = Counter(tok for toks in token_lists for tok in toks)
        for tok, freq in counter.most_common():
            if freq >= self.min_freq and tok not in self.stoi:
                idx = len(self.stoi)
                self.stoi[tok] = idx
                self.itos[idx] = tok
        print(f"  Vocab size: {len(self)}")

    def encode(self, tokens):
        return [self.stoi.get(t, self.unk_idx) for t in tokens]

    def decode(self, indices, skip_special=True):
        special = {self.pad_idx, self.sos_idx, self.eos_idx}
        out = []
        for idx in indices:
            if skip_special and idx in special:
                continue
            out.append(self.itos.get(idx, cfg.UNK))
        return out


In [5]:
def load_and_tokenize(path: str):
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Dataset file not found: {path}\n"
            f"Please place the CSV in the same folder as nmt_en_vi.py or update Config.TRAIN_FILE/VAL_FILE/TEST_FILE to the full path."
        )
    try:
        df = pd.read_csv(path)
    except Exception as e:
        raise RuntimeError(f"Failed to read CSV file {path}: {e}") from e

    df = df[["en", "vi"]].dropna()
    src_toks, tgt_toks = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Tokenising {os.path.basename(path)}"):
        en_tok = tokenize_en(str(row["en"]))[:cfg.MAX_LEN]
        vi_tok = tokenize_vi(str(row["vi"]))[:cfg.MAX_LEN]
        if en_tok and vi_tok:
            src_toks.append(en_tok)
            tgt_toks.append(vi_tok)
    return src_toks, tgt_toks


class TranslationDataset(Dataset):
    def __init__(self, src_toks, tgt_toks, src_vocab: Vocabulary, tgt_vocab: Vocabulary):
        self.pairs = []
        for s, t in zip(src_toks, tgt_toks):
            src_ids = src_vocab.encode(s)
            tgt_ids = [tgt_vocab.sos_idx] + tgt_vocab.encode(t) + [tgt_vocab.eos_idx]
            self.pairs.append((src_ids, tgt_ids))

    def __len__(self):  return len(self.pairs)
    def __getitem__(self, i): return self.pairs[i]


def collate_fn(batch, src_pad, tgt_pad):
    src_batch, tgt_batch = zip(*batch)
    src_lens = [len(s) for s in src_batch]
    tgt_lens = [len(t) for t in tgt_batch]

    max_src = max(src_lens)
    max_tgt = max(tgt_lens)

    src_pad_batch = torch.tensor(
        [s + [src_pad] * (max_src - len(s)) for s in src_batch], dtype=torch.long
    )
    tgt_pad_batch = torch.tensor(
        [t + [tgt_pad] * (max_tgt - len(t)) for t in tgt_batch], dtype=torch.long
    )
    src_lens = torch.tensor(src_lens, dtype=torch.long)
    return src_pad_batch, tgt_pad_batch, src_lens


In [6]:
# Encoder (BiLSTM) 
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.LSTM(
            emb_dim, hid_dim, n_layers,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True, bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        # project bidirectional hidden/cell → decoder hidden/cell size
        self.fc_h = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_c = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src, src_lens):
        # src      : (B, T_src)
        # src_lens : (B,)
        embedded = self.dropout(self.embedding(src))           # (B, T, E)

        packed = pack_padded_sequence(
            embedded, src_lens.cpu(), batch_first=True, enforce_sorted=False
        )
        outputs, (hidden, cell) = self.rnn(packed)
        outputs, _ = pad_packed_sequence(outputs, batch_first=True)
        # outputs : (B, T, 2*H)
        # hidden  : (2*layers, B, H)  →  need (layers, B, H) for decoder

        # Merge forward + backward for each layer
        def merge(state):
            # state: (2*layers, B, H)
            layers = state.shape[0] // 2
            merged = torch.tanh(
                torch.cat([state[2*i:2*i+2].transpose(0,1).contiguous().view(
                    state.shape[1], -1) for i in range(layers)], dim=0
                ).view(layers, state.shape[1], -1)
            )   # rough stacking — use linear projection instead:
            # simpler: project each layer separately
            return state

        # Cleaner approach: project last bidir hidden/cell to dec size
        # hidden: (2*enc_layers, B, H)  → take top forward+backward
        def proj_state(fc, state):
            # state: (n_dir*n_layers, B, H)
            # Stack pairs: layer0_fwd, layer0_bwd, layer1_fwd, layer1_bwd …
            n = state.shape[0] // 2     # number of layers
            out = []
            for i in range(n):
                fwd = state[2*i]         # (B, H)
                bwd = state[2*i + 1]     # (B, H)
                out.append(torch.tanh(fc(torch.cat([fwd, bwd], dim=-1))))  # (B, H)
            return torch.stack(out, dim=0)   # (n_layers, B, H)

        hidden = proj_state(self.fc_h, hidden)   # (enc_layers, B, H)
        cell   = proj_state(self.fc_c, cell)

        return outputs, hidden, cell


# Bahdanau Attention 
class BahdanauAttention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.W_h = nn.Linear(hid_dim, hid_dim, bias=False)   # decoder hidden
        self.W_s = nn.Linear(hid_dim * 2, hid_dim, bias=False)  # encoder output (bidir)
        self.v   = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs, src_mask=None):
        """
        dec_hidden  : (B, H)          — top-layer decoder hidden at t
        enc_outputs : (B, T_src, 2H)  — all encoder states
        src_mask    : (B, T_src) bool  — True where PAD
        returns:
          context   : (B, 2H)
          attn_w    : (B, T_src)
        """
        T = enc_outputs.shape[1]
        dec_h = self.W_h(dec_hidden).unsqueeze(1).expand(-1, T, -1)  # (B, T, H)
        enc_h = self.W_s(enc_outputs)                                  # (B, T, H)
        energy = self.v(torch.tanh(dec_h + enc_h)).squeeze(-1)        # (B, T)

        if src_mask is not None:
            energy = energy.masked_fill(src_mask, float("-inf"))

        attn_w  = F.softmax(energy, dim=-1)                           # (B, T)
        context = torch.bmm(attn_w.unsqueeze(1), enc_outputs).squeeze(1)  # (B, 2H)
        return context, attn_w


# Decoder (LSTM + Attention) 
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.attention = BahdanauAttention(hid_dim)
        # input to LSTM: emb + context (2H)
        self.rnn = nn.LSTM(
            emb_dim + hid_dim * 2, hid_dim, n_layers,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        self.fc_out = nn.Linear(hid_dim * 3 + emb_dim, vocab_size)   # Luong-style
        self.dropout = nn.Dropout(dropout)

    def forward_step(self, tgt_tok, hidden, cell, enc_outputs, src_mask):
        """One decoding step."""
        embedded = self.dropout(self.embedding(tgt_tok.unsqueeze(1)))  # (B,1,E)
        dec_top  = hidden[-1]                                           # (B, H)
        context, attn_w = self.attention(dec_top, enc_outputs, src_mask)  # (B,2H)

        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=-1)   # (B,1,E+2H)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))      # (B,1,H)
        output = output.squeeze(1)                                          # (B, H)

        pred = self.fc_out(
            torch.cat([output, context, embedded.squeeze(1)], dim=-1)
        )                                                                   # (B, V)
        return pred, hidden, cell, attn_w


# Seq2Seq  
class Seq2Seq(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, src_pad_idx, tgt_pad_idx):
        super().__init__()
        self.encoder     = encoder
        self.decoder     = decoder
        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx

    def make_src_mask(self, src):
        return (src == self.src_pad_idx)   # (B, T)  True at PAD

    def forward(self, src, src_lens, tgt, teacher_forcing_ratio=0.5):
        """
        src  : (B, T_src)
        tgt  : (B, T_tgt)   including <sos> and <eos>
        """
        B, T_tgt = tgt.shape
        vocab_sz = self.decoder.fc_out.out_features

        enc_outputs, hidden, cell = self.encoder(src, src_lens)
        src_mask = self.make_src_mask(src)   # (B, T_src)

        # Repeat encoder layers if enc_layers < dec_layers
        enc_layers = hidden.shape[0]
        dec_layers = cfg.DEC_LAYERS
        if enc_layers < dec_layers:
            hidden = hidden.repeat(math.ceil(dec_layers / enc_layers), 1, 1)[:dec_layers]
            cell   = cell.repeat(math.ceil(dec_layers / enc_layers), 1, 1)[:dec_layers]

        outputs  = torch.zeros(B, T_tgt - 1, vocab_sz, device=src.device)
        dec_input = tgt[:, 0]   # <sos>

        for t in range(T_tgt - 1):
            pred, hidden, cell, _ = self.decoder.forward_step(
                dec_input, hidden, cell, enc_outputs, src_mask
            )
            outputs[:, t] = pred
            top1 = pred.argmax(-1)
            use_teacher = random.random() < teacher_forcing_ratio
            dec_input = tgt[:, t + 1] if use_teacher else top1

        return outputs   # (B, T_tgt-1, V)


In [7]:
def train_epoch(model, loader, optimizer, criterion, clip):
    model.train()
    total_loss = 0.0
    print(f"  train_epoch: batches={len(loader):,}")
    for batch_idx, (src, tgt, src_lens) in enumerate(tqdm(loader, desc="  Train", unit="it", leave=False, ncols=100)):
        if batch_idx == 0:
            print(f"    first batch shape: src={src.shape} tgt={tgt.shape}")
        src, tgt, src_lens = src.to(cfg.DEVICE), tgt.to(cfg.DEVICE), src_lens
        optimizer.zero_grad()

        output = model(src, src_lens, tgt, cfg.TEACHER_FORCING)
        # output: (B, T-1, V)   tgt[:,1:]: (B, T-1)
        output_flat = output.reshape(-1, output.shape[-1])
        tgt_flat    = tgt[:, 1:].reshape(-1)

        loss = criterion(output_flat, tgt_flat)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    for src, tgt, src_lens in tqdm(loader, desc="  Val  ", unit="it", leave=False, ncols=100):
        src, tgt = src.to(cfg.DEVICE), tgt.to(cfg.DEVICE)
        output = model(src, src_lens, tgt, teacher_forcing_ratio=0.0)
        output_flat = output.reshape(-1, output.shape[-1])
        tgt_flat    = tgt[:, 1:].reshape(-1)
        total_loss += criterion(output_flat, tgt_flat).item()
    return total_loss / len(loader)


In [8]:
# Beam search decoding
@torch.no_grad()
def beam_search(model, src_ids: list, src_vocab, tgt_vocab,
                beam_size=5, max_len=100):
    model.eval()
    src = torch.tensor([src_ids], dtype=torch.long, device=cfg.DEVICE)
    src_lens = torch.tensor([len(src_ids)])

    enc_outputs, hidden, cell = model.encoder(src, src_lens)
    src_mask = model.make_src_mask(src)

    # Adjust hidden/cell layers
    enc_layers = hidden.shape[0]
    if enc_layers < cfg.DEC_LAYERS:
        hidden = hidden.repeat(math.ceil(cfg.DEC_LAYERS / enc_layers), 1, 1)[:cfg.DEC_LAYERS]
        cell   = cell.repeat(math.ceil(cfg.DEC_LAYERS / enc_layers), 1, 1)[:cfg.DEC_LAYERS]

    # beam entry: (log_prob, token_ids, hidden, cell)
    beams = [(0.0, [tgt_vocab.sos_idx], hidden, cell)]
    completed = []

    for _ in range(max_len):
        all_candidates = []
        for log_p, seq, h, c in beams:
            last_tok = torch.tensor([seq[-1]], device=cfg.DEVICE)
            pred, h_new, c_new, _ = model.decoder.forward_step(
                last_tok, h, c, enc_outputs, src_mask
            )
            log_probs = F.log_softmax(pred[0], dim=-1)
            topk_lp, topk_ids = log_probs.topk(beam_size)
            for lp, tid in zip(topk_lp.tolist(), topk_ids.tolist()):
                all_candidates.append((log_p + lp, seq + [tid], h_new, c_new))

        all_candidates.sort(key=lambda x: x[0], reverse=True)
        beams = []
        for cand in all_candidates:
            if cand[1][-1] == tgt_vocab.eos_idx:
                completed.append(cand)
            else:
                beams.append(cand)
            if len(beams) == beam_size:
                break
        if not beams:
            break

    if completed:
        completed.sort(key=lambda x: x[0] / len(x[1]), reverse=True)
        best = completed[0][1]
    else:
        beams.sort(key=lambda x: x[0] / len(x[1]), reverse=True)
        best = beams[0][1]

    return tgt_vocab.decode(best)

# BLEU evaluation
@torch.no_grad()
def compute_bleu(model, dataset_pairs, src_vocab, tgt_vocab, n_samples=500):
    model.eval()
    hypotheses, references = [], []
    indices = random.sample(range(len(dataset_pairs)), min(n_samples, len(dataset_pairs)))
    for i in tqdm(indices, desc="  BLEU"):
        src_ids, tgt_ids = dataset_pairs[i]
        hyp = beam_search(model, src_ids, src_vocab, tgt_vocab,
                          beam_size=cfg.BEAM_SIZE, max_len=cfg.MAX_DECODE)
        ref = tgt_vocab.decode(tgt_ids)
        hypotheses.append(" ".join(hyp))
        references.append(" ".join(ref))
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return bleu.score



In [ ]:
def main():
    # Load & tokenise 
    print("\n[1/5] Loading & tokenising data …")
    train_src, train_tgt = load_and_tokenize(cfg.TRAIN_FILE)
    val_src,   val_tgt   = load_and_tokenize(cfg.VAL_FILE)
    test_src,  test_tgt  = load_and_tokenize(cfg.TEST_FILE)

    # Build vocabularies
    print("\n[2/5] Building vocabularies …")
    src_vocab = Vocabulary(min_freq=cfg.MIN_FREQ)
    tgt_vocab = Vocabulary(min_freq=cfg.MIN_FREQ)
    print("  Source (EN):")
    src_vocab.build(train_src)
    print("  Target (VI):")
    tgt_vocab.build(train_tgt)

    # Datasets & loaders 
    print("\n[3/5] Building datasets …")
    from functools import partial
    collate = partial(collate_fn,
                      src_pad=src_vocab.pad_idx,
                      tgt_pad=tgt_vocab.pad_idx)

    train_ds = TranslationDataset(train_src, train_tgt, src_vocab, tgt_vocab)
    val_ds   = TranslationDataset(val_src,   val_tgt,   src_vocab, tgt_vocab)
    test_ds  = TranslationDataset(test_src,  test_tgt,  src_vocab, tgt_vocab)

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,
                              shuffle=True,  collate_fn=collate, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE,
                              shuffle=False, collate_fn=collate, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE,
                              shuffle=False, collate_fn=collate, num_workers=0)

    print(f"  Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")
    print(f"  Train batches: {len(train_loader):,}  Val batches: {len(val_loader):,}  Test batches: {len(test_loader):,}")

    # Build model 
    print("\n[4/5] Building model …")
    encoder = Encoder(
        vocab_size=len(src_vocab),
        emb_dim=cfg.EMB_DIM,
        hid_dim=cfg.HID_DIM,
        n_layers=cfg.ENC_LAYERS,
        dropout=cfg.DROPOUT,
        pad_idx=src_vocab.pad_idx,
    )
    decoder = Decoder(
        vocab_size=len(tgt_vocab),
        emb_dim=cfg.EMB_DIM,
        hid_dim=cfg.HID_DIM,
        n_layers=cfg.DEC_LAYERS,
        dropout=cfg.DROPOUT,
        pad_idx=tgt_vocab.pad_idx,
    )
    model = Seq2Seq(encoder, decoder,
                    src_pad_idx=src_vocab.pad_idx,
                    tgt_pad_idx=tgt_vocab.pad_idx).to(cfg.DEVICE)

    def count_params(m):
        return sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {count_params(model):,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)

    # Training loop
    print("\n[5/5] Training …")
    best_val_loss = float("inf")
    history = []

    for epoch in range(1, cfg.EPOCHS + 1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, cfg.CLIP_GRAD)
        val_loss   = evaluate(model, val_loader, criterion)
        val_bleu   = compute_bleu(model, val_ds.pairs, src_vocab, tgt_vocab, n_samples=200)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        train_ppl = math.exp(train_loss)
        val_ppl   = math.exp(val_loss)
        history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                            train_ppl=train_ppl, val_ppl=val_ppl, val_bleu=val_bleu))

        flag = ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pt")
            flag = "  ← saved"

        print(
            f"Epoch {epoch:02d}/{cfg.EPOCHS} | "
            f"train_loss={train_loss:.4f} ppl={train_ppl:.2f} | "
            f"val_loss={val_loss:.4f} ppl={val_ppl:.2f} | "
            f"val_bleu={val_bleu:.2f} | "
            f"time={elapsed:.1f}s{flag}"
        )

        # Plot learning curves
        print("Plotting learning curves …")
        epochs = [h["epoch"] for h in history]
        train_losses = [h["train_loss"] for h in history]
        val_losses = [h["val_loss"] for h in history]
        val_bleus = [h["val_bleu"] for h in history]

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(epochs, train_losses, marker="o", label="Train Loss")
        plt.plot(epochs, val_losses, marker="o", label="Val Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Loss Curve")
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(epochs, val_bleus, marker="o", color="tab:orange", label="Val BLEU")
        plt.xlabel("Epoch")
        plt.ylabel("BLEU")
        plt.title("Validation BLEU Curve")
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.show()

    # Test evaluation
    print("\nLoading best checkpoint …")
    model.load_state_dict(torch.load("best_model.pt", map_location=cfg.DEVICE))
    test_loss = evaluate(model, test_loader, criterion)
    print(f"Test loss: {test_loss:.4f} | PPL: {math.exp(test_loss):.2f}")

    print("\nComputing BLEU on test set (500 samples, beam search) …")
    bleu = compute_bleu(model, test_ds.pairs, src_vocab, tgt_vocab, n_samples=500)
    print(f"BLEU score: {bleu:.2f}")

    # Demo translations 
    demo_sentences = [
        "I love you.",
        "How are you doing today?",
        "The weather is beautiful outside.",
        "She went to the market to buy vegetables.",
    ]
    print("\n─── Demo translations ───")
    for sent in demo_sentences:
        src_ids = src_vocab.encode(tokenize_en(sent)[:cfg.MAX_LEN])
        out = beam_search(model, src_ids, src_vocab, tgt_vocab,
                          beam_size=cfg.BEAM_SIZE)
        print(f"EN: {sent}")
        print(f"VI: {' '.join(out)}\n")

    # Save training history
    pd.DataFrame(history).to_csv("training_history.csv", index=False)
    print("Saved training_history.csv")


if __name__ == "__main__":
    main()

NameError: name 'train_loss' is not defined